# FAF 5.7.1 Freight Flows — Phase 1 Profiling (PR#1)

This notebook is a **portfolio-facing profiling artifact** for a MySQL freight-flow warehouse built from the U.S. DOT/BTS **Freight Analysis Framework (FAF) 5.7.1** (years **2018–2024**).

## What a reviewer should take away
- The raw file is **large** (hundreds of MB) → we profile with **chunked reads** (memory-safe).
- The dataset is **wide** across multiple measure families using the pattern: `<measure_family>_<year>`.
- Core key columns (zones, modes, commodity, trade type, distance band) are identified with approximate cardinalities.
- Using **official FAF metadata**, we confirm `dms_*` (domestic FAF zones) and `fr_*` (foreign regions) are **two different zone systems** (not duplicates).
- We define the **target star-schema grain** (fact table grain + measures) that will drive ETL, validation gates, and indexing in later PRs.

> Raw CSV is intentionally **excluded from Git**. Place it locally at: `data/raw/faf/FAF5.7.1_2018-2024.csv`


## 0) Reproducibility notes
- This notebook uses **relative paths** so it runs after cloning the repo.
- It avoids full in-memory loads and only reads: (a) a small head sample, (b) a first chunk, and (c) chunk scans for row counts.


## 1) Setup (imports, paths, and chunk strategy)


In [1]:
from pathlib import Path
import re
import pandas as pd
import numpy as np
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

CSV_PATH = Path('../data/raw/faf/FAF5.7.1_2018-2024.csv')
META_XLSX = Path('../data/metadata/faf/FAF5_metadata.xlsx')

# Profiling chunk size: safe default on most laptops.
# For MySQL loads later, we will tune read chunk size and insert batch size separately.
CHUNK_SIZE = 200_000

assert CSV_PATH.exists(), f'Raw CSV not found: {CSV_PATH.resolve()}'
assert META_XLSX.exists(), f'Metadata XLSX not found: {META_XLSX.resolve()}'

csv_size_mb = CSV_PATH.stat().st_size / (1024**2)
print(f'CSV size: {csv_size_mb:,.1f} MB')
print('Chunk size:', CHUNK_SIZE)


CSV size: 663.0 MB
Chunk size: 200000


## 2) Quick preview (schema glimpse, no full load)
We read only the first few rows to confirm column names and overall structure.


In [2]:
df_head = pd.read_csv(CSV_PATH, nrows=10)
display(df_head.head())
cols = df_head.columns.tolist()
print('Columns:', len(cols))


,fr_orig,dms_orig,dms_dest,fr_dest,fr_inmode,dms_mode,fr_outmode,sctg2,trade_type,dist_band,tons_2018,tons_2019,tons_2020,tons_2021,tons_2022,tons_2023,tons_2024,value_2018,value_2019,value_2020,value_2021,value_2022,value_2023,value_2024,current_value_2018,current_value_2019,current_value_2020,current_value_2021,current_value_2022,current_value_2023,current_value_2024,tmiles_2018,tmiles_2019,tmiles_2020,tmiles_2021,tmiles_2022,tmiles_2023,tmiles_2024
0,NaN,11,11,NaN,NaN,1,NaN,1,1,1,51.863434,54.012407,54.395645,56.117256,56.434027,55.902481,55.990230,68.594362,71.436584,71.943453,74.220449,74.639408,73.936389,74.052445,66.560261,67.993017,62.808675,81.560852,101.069223,104.272629,115.502844,3.218410,3.351765,3.375547,3.482383,3.502040,3.469055,3.474500
1,NaN,11,19,NaN,NaN,1,NaN,1,1,2,392.072310,408.317910,411.215079,424.229954,426.624644,422.606320,423.269675,518.553218,540.039582,543.871367,561.084786,564.251994,558.937374,559.814725,503.176016,514.007225,474.815125,616.576072,764.053625,788.270430,873.167561,48.631191,50.646235,51.005589,52.619906,52.916934,52.418516,52.500796
2,NaN,11,129,NaN,NaN,1,NaN,1,1,3,1.383202,1.440515,1.450736,1.496652,1.505100,1.490924,1.493264,1.829418,1.905220,1.918738,1.979466,1.990640,1.971890,1.974985,1.775168,1.813380,1.675113,2.175235,2.695525,2.780960,3.080471,0.450066,0.468714,0.472040,0.486980,0.489729,0.485116,0.485878
3,NaN,11,131,NaN,NaN,1,NaN,1,1,2,12.698528,13.224694,13.318528,13.740057,13.817617,13.687470,13.708955,16.795020,17.490925,17.615030,18.172542,18.275123,18.102991,18.131407,16.296980,16.647783,15.378421,19.969807,24.746343,25.530683,28.280351,2.177036,2.267241,2.283328,2.355595,2.368892,2.346580,2.350263
4,NaN,11,139,NaN,NaN,1,NaN,1,1,2,5.220302,5.436606,5.475181,5.648469,5.680354,5.626851,5.635684,6.904350,7.190433,7.241452,7.470643,7.512813,7.442051,7.453732,6.699608,6.843822,6.321993,8.209489,10.173100,10.495538,11.625913,1.274964,1.327792,1.337213,1.379536,1.387323,1.374256,1.376413


Columns: 38


### Column list (ground truth)


In [3]:
cols


['fr_orig',
 'dms_orig',
 'dms_dest',
 'fr_dest',
 'fr_inmode',
 'dms_mode',
 'fr_outmode',
 'sctg2',
 'trade_type',
 'dist_band',
 'tons_2018',
 'tons_2019',
 'tons_2020',
 'tons_2021',
 'tons_2022',
 'tons_2023',
 'tons_2024',
 'value_2018',
 'value_2019',
 'value_2020',
 'value_2021',
 'value_2022',
 'value_2023',
 'value_2024',
 'current_value_2018',
 'current_value_2019',
 'current_value_2020',
 'current_value_2021',
 'current_value_2022',
 'current_value_2023',
 'current_value_2024',
 'tmiles_2018',
 'tmiles_2019',
 'tmiles_2020',
 'tmiles_2021',
 'tmiles_2022',
 'tmiles_2023',
 'tmiles_2024']

## 3) Total row count (chunk scan)
We compute total rows with a chunk scan. This is memory-safe but can take a few minutes on large files.


In [4]:
row_count = 0
for chunk in pd.read_csv(CSV_PATH, chunksize=CHUNK_SIZE, usecols=[cols[0]]):
    row_count += len(chunk)
row_count


2494901

## 4) Year + measure structure (wide across measure families)

FAF flows in this file are stored in **wide** format using the pattern:

`<measure_family>_<year>`

Example families typically include: `tons`, `value`, `current_value`, `tmiles`.

Below we detect all columns matching this pattern for years **2018–2024**, then validate year coverage per family.


In [5]:
YEAR_MIN, YEAR_MAX = 2018, 2024
valid_years = {str(y) for y in range(YEAR_MIN, YEAR_MAX + 1)}

# Generic detector: 'something_2019'
pat = re.compile(r'^(?P<measure>[A-Za-z_]+)_(?P<year>20\d{2})$')

measure_map = {}  # family -> set(years)
measure_cols = []
for c in cols:
    m = pat.match(str(c))
    if not m:
        continue
    year = m.group('year')
    if year not in valid_years:
        continue
    fam = m.group('measure').lower()
    measure_map.setdefault(fam, set()).add(year)
    measure_cols.append(c)

print('Detected measure families:', sorted(measure_map.keys()))
print('Detected year-suffixed measure columns:', len(measure_cols))


Detected measure families: ['current_value', 'tmiles', 'tons', 'value']
Detected year-suffixed measure columns: 28


### Year coverage per measure family


In [6]:
coverage = (
    pd.DataFrame([
        {
            'measure_family': fam,
            'n_years_found': len(yrs),
            'years_found': ', '.join(sorted(list(yrs))),
            'missing_years': ', '.join(sorted(list(valid_years - set(yrs))))
        }
        for fam, yrs in sorted(measure_map.items())
    ])
    .sort_values(['n_years_found','measure_family'], ascending=[False, True])
)
display(coverage)


,measure_family,n_years_found,years_found,missing_years
0,current_value,7,"2018, 2019, 2020, 2021, 2022, 2023, 2024",
1,tmiles,7,"2018, 2019, 2020, 2021, 2022, 2023, 2024",
2,tons,7,"2018, 2019, 2020, 2021, 2022, 2023, 2024",
3,value,7,"2018, 2019, 2020, 2021, 2022, 2023, 2024",


**Interpretation:** families with **7 years found** and **no missing years** are complete for 2018–2024.
Later ETL will reshape the dataset into long form aligned by `year`.


## 5) Data-quality snapshot (first chunk)


In [7]:
first_chunk = next(pd.read_csv(CSV_PATH, chunksize=CHUNK_SIZE))
first_chunk.shape


(200000, 38)

### Missingness (top 20 columns by null %)


In [8]:
(first_chunk.isna().mean() * 100).sort_values(ascending=False).head(20)


fr_orig       100.0
fr_dest       100.0
fr_inmode     100.0
fr_outmode    100.0
dms_dest        0.0
dms_orig        0.0
dms_mode        0.0
sctg2           0.0
trade_type      0.0
dist_band       0.0
tons_2018       0.0
tons_2019       0.0
tons_2020       0.0
tons_2021       0.0
tons_2022       0.0
tons_2023       0.0
tons_2024       0.0
value_2018      0.0
value_2019      0.0
value_2020      0.0
dtype: float64

### Interpretation of missingness

The `fr_*` fields show 100% null in this first chunk.

This is expected when the sampled records represent purely domestic freight flows.
For domestic movements, only `dms_*` zone fields are populated.

Later in this notebook, multi-chunk sampling confirms that foreign flows exist and that
`dms_*` and `fr_*` behave as complementary zone systems rather than redundant fields.


In [9]:
# Improved missingness check for code columns (treat 0 as missing)

code_columns = [
    'dms_orig','dms_dest',
    'fr_orig','fr_dest',
    'dms_mode','fr_inmode','fr_outmode',
    'sctg2','trade_type','dist_band'
]

existing_code_cols = [c for c in code_columns if c in first_chunk.columns]

def missing_with_zero(series):
    if pd.api.types.is_numeric_dtype(series):
        return ((series.isna()) | (series == 0)).mean() * 100
    else:
        s = series.astype(str).str.strip().str.lower()
        return ((series.isna()) | (s.isin(['0','0.0','', 'nan', 'none']))).mean() * 100

improved_missingness = {
    col: missing_with_zero(first_chunk[col])
    for col in existing_code_cols
}

pd.Series(improved_missingness).sort_values(ascending=False)


fr_dest       100.0
fr_orig       100.0
fr_inmode     100.0
fr_outmode    100.0
dms_dest        0.0
dms_orig        0.0
dms_mode        0.0
sctg2           0.0
trade_type      0.0
dist_band       0.0
dtype: float64

### Interpretation (code columns)

This view treats both `NaN` and `0` values as missing for categorical code fields.

This confirms that high missingness in `fr_*` is structural (domestic-only flows)
rather than data corruption.


### Duplicate rows (within first chunk)


In [10]:
dup_count = int(first_chunk.duplicated().sum())
dup_count


0

### Negative values (year-suffixed measure columns, first chunk)


In [11]:
if len(measure_cols) == 0:
    neg_total = 0
else:
    numeric_measures = first_chunk[measure_cols].apply(pd.to_numeric, errors='coerce')
    neg_total = int((numeric_measures < 0).sum().sum())
neg_total


0

### Representativeness check (more than one chunk)

The quality checks above use the first chunk for speed. To reduce the risk of “first-chunk bias” in a large file,
we also sample two additional chunks:

- a **middle** chunk (approximate),
- the **last** chunk.

We repeat lightweight checks on each sample:
- key-column null counts,
- negative values across year-suffixed measures,
- DMS vs FR complementarity (origin/destination).

> **Runtime note:** reading the middle/last chunks is still memory-safe, but it may take **1–3 minutes** on slower disks because it must iterate through chunks.



In [12]:
# --- Ensure key columns are defined for this section (self-contained) ---
key_cols = ['dms_orig','dms_dest','fr_orig','fr_dest','dms_mode','fr_inmode','fr_outmode','sctg2','trade_type','dist_band']
present_keys = [c for c in key_cols if c in first_chunk.columns]

# Helper: read specific chunks without loading full file
def read_nth_chunk(n: int):
    for i, ch in enumerate(pd.read_csv(CSV_PATH, chunksize=CHUNK_SIZE)):
        if i == n:
            return ch
    raise ValueError(f'Requested chunk {n} out of range')

# Determine total number of chunks (fast-ish scan reading 1 column only)
total_rows = row_count  # computed earlier
n_chunks = int(np.ceil(total_rows / CHUNK_SIZE))
mid_chunk_idx = n_chunks // 2
last_chunk_idx = max(n_chunks - 1, 0)
print('Estimated chunks:', n_chunks, 'mid:', mid_chunk_idx, 'last:', last_chunk_idx)

chunks_to_check = {
    'first': first_chunk,
    'mid'  : read_nth_chunk(mid_chunk_idx) if n_chunks > 1 else first_chunk,
    'last' : read_nth_chunk(last_chunk_idx) if n_chunks > 1 else first_chunk,
}

def chunk_checks(name, ch):
    # negatives
    if len(measure_cols) == 0:
        neg = 0
    else:
        nm = ch[measure_cols].apply(pd.to_numeric, errors='coerce')
        neg = int((nm < 0).sum().sum())

    # key nulls
    key_nulls = ch[present_keys].isna().sum().sort_values(ascending=False) if present_keys else pd.Series(dtype=int)

    # dms vs fr complementarity
    def present_code(series):
        s = series
        mask = ~s.isna()
        if pd.api.types.is_numeric_dtype(s):
            return mask & (s != 0)
        st = s.astype(str).str.strip().str.lower()
        return mask & (~st.isin(['', '0', '0.0', 'nan', 'none']))

    if 'dms_orig' in ch.columns and 'fr_orig' in ch.columns:
        dms_o = present_code(ch['dms_orig'])
        fr_o  = present_code(ch['fr_orig'])
        origin = pd.Series({
            'both': int((dms_o & fr_o).sum()),
            'only_dms': int((dms_o & ~fr_o).sum()),
            'only_fr': int((~dms_o & fr_o).sum()),
            'neither': int((~dms_o & ~fr_o).sum()),
        })
    else:
        origin = pd.Series(dtype=int)

    if 'dms_dest' in ch.columns and 'fr_dest' in ch.columns:
        dms_d = present_code(ch['dms_dest'])
        fr_d  = present_code(ch['fr_dest'])
        dest = pd.Series({
            'both': int((dms_d & fr_d).sum()),
            'only_dms': int((dms_d & ~fr_d).sum()),
            'only_fr': int((~dms_d & fr_d).sum()),
            'neither': int((~dms_d & ~fr_d).sum()),
        })
    else:
        dest = pd.Series(dtype=int)

    return {
        'negatives': neg,
        'key_nulls_top5': key_nulls.head(5),
        'origin_overlap': origin,
        'dest_overlap': dest,
        'rows_in_chunk': len(ch),
    }

results = {name: chunk_checks(name, ch) for name, ch in chunks_to_check.items()}

neg_table = pd.DataFrame({k: {'negatives': v['negatives'], 'rows_in_chunk': v['rows_in_chunk']} for k,v in results.items()}).T
display(neg_table)

print('\nTop-5 key null counts (per sample):')
for k,v in results.items():
    print(f'\n[{k}]')
    display(v['key_nulls_top5'].to_frame('null_count'))

print('\nDMS vs FR overlap (origin/destination) per sample:')
for k,v in results.items():
    print(f'\n[{k}] origin')
    display(v['origin_overlap'].to_frame('count'))
    print(f'[{k}] destination')
    display(v['dest_overlap'].to_frame('count'))


Estimated chunks: 13 mid: 6 last: 12


,negatives,rows_in_chunk
first,0,200000
mid,0,200000
last,0,94901



Top-5 key null counts (per sample):

[first]


,null_count
fr_dest,200000
fr_orig,200000
fr_inmode,200000
fr_outmode,200000
dms_dest,0



[mid]


...

,count
both,0
only_dms,200000
only_fr,0
neither,0



[last] origin


,count
both,0
only_dms,94901
only_fr,0
neither,0


[last] destination


,count
both,94901
only_dms,0
only_fr,0
neither,0


### Representativeness Conclusion

The first chunk appears dominated by domestic flows (high `dms_*`, null `fr_*`).

Middle and last chunks confirm the presence of foreign flows,
demonstrating that `dms_*` and `fr_*` are complementary systems.

This validates the decision to unify both into a single `dim_zone`
with a `zone_type` attribute in later phases.


## 6) Dimension candidates (keys) and rough cardinalities
These columns define the future star schema joins and are used to build dimension tables.


In [13]:
key_cols = ['dms_orig','dms_dest','fr_orig','fr_dest','dms_mode','fr_inmode','fr_outmode','sctg2','trade_type','dist_band']
present_keys = [c for c in key_cols if c in first_chunk.columns]
print('Key columns present:', present_keys)


Key columns present: ['dms_orig', 'dms_dest', 'fr_orig', 'fr_dest', 'dms_mode', 'fr_inmode', 'fr_outmode', 'sctg2', 'trade_type', 'dist_band']


### Approximate unique counts (first chunk)


In [14]:
card = pd.Series({c: first_chunk[c].nunique(dropna=True) for c in present_keys}).sort_values(ascending=False)
display(card.to_frame('unique_in_first_chunk'))


,unique_in_first_chunk
dms_orig,132
dms_dest,132
sctg2,29
dist_band,8
dms_mode,7
trade_type,1
fr_inmode,0
fr_dest,0
fr_orig,0
fr_outmode,0


### Null counts on key columns (first chunk)


In [15]:
display(first_chunk[present_keys].isna().sum().sort_values(ascending=False).to_frame('null_count_first_chunk'))


,null_count_first_chunk
fr_dest,200000
fr_orig,200000
fr_inmode,200000
fr_outmode,200000
dms_dest,0
dms_orig,0
dms_mode,0
sctg2,0
trade_type,0
dist_band,0


### Note on mode fields (`dms_mode` vs `fr_inmode` / `fr_outmode`)

This file contains multiple mode-related fields:
- `dms_mode` is the primary mode code for domestic FAF flows.
- `fr_inmode` / `fr_outmode` can represent international legs (e.g., inbound/outbound portions of foreign trade flows).

**PR#1 scope:** we only identify these as candidates.  
**Next PRs:** we will define a single `dim_mode` strategy (mapping rules + primary key) so analytics can use one consistent mode dimension.


## 7) `dms_*` vs `fr_*` zones — metadata-backed interpretation

**Question:** Are `fr_orig`/`fr_dest` redundant with `dms_orig`/`dms_dest`?  
**Answer:** No. The metadata defines **two different zone systems**:
- `dms_*` → domestic FAF zones (and/or state mapping)
- `fr_*` → foreign regions

We show the definitions from the official metadata and validate complementarity empirically on a chunk.


In [16]:
# Read the Data Dictionary. The sheet has an internal header row (Field / Description / Link (Tab)).
dd_raw = pd.read_excel(META_XLSX, sheet_name='Data Dictionary')
first_col = dd_raw.columns[0]
header_row_idx = dd_raw.index[dd_raw[first_col].astype(str).str.strip().eq('Field')].tolist()
assert header_row_idx, 'Could not locate the Data Dictionary header row.'
header_row_idx = header_row_idx[0]

dd = dd_raw.iloc[header_row_idx:].copy()
dd.columns = dd.iloc[0].tolist()
dd = dd.iloc[1:].reset_index(drop=True)

# Normalize column names (some headers include trailing spaces); keep any extra column as 'Extra'
dd.columns = [str(c).strip() if not pd.isna(c) else 'Extra' for c in dd.columns]
dd.head()


,Field,Description,Link (Tab),Extra
0,fr_orig,Foreign region of shipment origin,FAF Zone (Foreign),NaN
1,dms_orig,FAF region or state where a freight movement b...,FAF Zone (Domestic),State
2,dms_dest,FAF region or state where a freight movement e...,FAF Zone (Domestic),State
3,fr_dest,Foreign region of shipment destination,FAF Zone (Foreign),NaN
4,sctg2,2-digit level of the Standard Classification o...,Commodity (SCTG2),NaN


### Zone field definitions (from metadata)


In [17]:
zone_fields = ['fr_orig','dms_orig','dms_dest','fr_dest']
cols_needed = [c for c in ['Field','Description','Link (Tab)','Extra'] if c in dd.columns]
defs = dd[dd['Field'].isin(zone_fields)][cols_needed].copy()
display(defs)


,Field,Description,Link (Tab),Extra
0,fr_orig,Foreign region of shipment origin,FAF Zone (Foreign),NaN
1,dms_orig,FAF region or state where a freight movement b...,FAF Zone (Domestic),State
2,dms_dest,FAF region or state where a freight movement e...,FAF Zone (Domestic),State
3,fr_dest,Foreign region of shipment destination,FAF Zone (Foreign),NaN


### Empirical check (first chunk): complementarity of DMS vs FR codes


In [18]:
def present_code(series):
    """Return boolean mask for 'code is present'. Treat NaN/blank as missing and 0/0.0 as missing."""
    s = series
    mask = ~s.isna()
    if pd.api.types.is_numeric_dtype(s):
        return mask & (s != 0)
    st = s.astype(str).str.strip().str.lower()
    return mask & (~st.isin(['', '0', '0.0', 'nan', 'none']))

has_dms_o = present_code(first_chunk['dms_orig']) if 'dms_orig' in first_chunk.columns else None
has_fr_o  = present_code(first_chunk['fr_orig'])  if 'fr_orig'  in first_chunk.columns else None
has_dms_d = present_code(first_chunk['dms_dest']) if 'dms_dest' in first_chunk.columns else None
has_fr_d  = present_code(first_chunk['fr_dest'])  if 'fr_dest'  in first_chunk.columns else None

def overlap(a, b):
    both = int((a & b).sum())
    only_a = int((a & ~b).sum())
    only_b = int((~a & b).sum())
    neither = int((~a & ~b).sum())
    return pd.Series({'both_present': both, 'only_dms': only_a, 'only_fr': only_b, 'neither': neither})

origin_overlap = overlap(has_dms_o, has_fr_o) if has_dms_o is not None else None
dest_overlap   = overlap(has_dms_d, has_fr_d) if has_dms_d is not None else None

display(pd.DataFrame({'origin(dms_vs_fr)': origin_overlap, 'dest(dms_vs_fr)': dest_overlap}))


,origin(dms_vs_fr),dest(dms_vs_fr)
both_present,0,0
only_dms,200000,200000
only_fr,0,0
neither,0,0


### Decision: unify zone systems in `dim_zone`
For a clean star schema we will consolidate both systems into one dimension:
- `dim_zone(zone_id, zone_type, zone_code, zone_name, …)` where `zone_type ∈ {DMS, FR}`
- the fact table stores `origin_zone_id` and `destination_zone_id`


## 8) Measure definitions from metadata (interpretability)


In [19]:
measure_fields = ['tons','value','current_value','tmiles']
cols_needed = [c for c in ['Field','Description','Link (Tab)'] if c in dd.columns]
defs_meas = dd[dd['Field'].isin(measure_fields)][cols_needed].copy()
display(defs_meas)


,Field,Description,Link (Tab)
10,tons,Total weight of commodities shipped (unit: Tho...,NaN
11,value,Total value (in 2017 constant dollar) of commo...,NaN
12,tmiles,Total ton-miles of commodities shipped (unit: ...,NaN
15,current_value,Total value (in current dollar of each year) o...,NaN


## Final schema summary (quick scan)

| Layer | What | Fields |
|---|---|---|
| **Fact grain** | One row per flow per year | `origin_zone_id`, `destination_zone_id`, `mode_id`, `commodity_id`, `trade_type_id`, `dist_band_id`, `year` |
| **Measures** | Stored on the fact | `tons`, `value` (2017 constant $), `current_value` (nominal $), `tmiles` |
| **Key raw fields** | Used to build dims / derive keys | `dms_orig`, `dms_dest`, `fr_orig`, `fr_dest`, `dms_mode`, `sctg2`, `trade_type`, `dist_band` |
| **Zone rule** | Metadata-backed | `dms_*` = domestic FAF zones, `fr_*` = foreign regions → unified into `dim_zone(zone_type)` |


## 9) Target warehouse grain (PR#1 deliverable)

### Target fact grain
One row per:

`(origin_zone_id, destination_zone_id, mode_id, commodity_id, trade_type_id, dist_band_id, year)`

### Fact measures
- `tons`
- `value` (2017 constant dollars, per metadata)
- `current_value` (current dollars of each year, per metadata)
- `tmiles`

### Transformation required
The raw file is wide across years. Later ETL will:
1) reshape columns like `tons_YYYY`, `value_YYYY`, `current_value_YYYY`, `tmiles_YYYY` into long form aligned by `year`
2) load a single multi-measure fact table at the grain above


In [20]:
expected_long_upper = row_count * len(valid_years)
print('Expected long rows (upper bound):', f'{expected_long_upper:,}')
print('Note: actual rows may be lower if year values are missing for some flows.')


Expected long rows (upper bound): 17,464,307
Note: actual rows may be lower if year values are missing for some flows.


## 10) Optional: create a small local sample for dev/testing (ignored by Git)


In [21]:
sample_path = Path('../data/interim/faf_sample_2000.csv')
sample_path.parent.mkdir(parents=True, exist_ok=True)
pd.read_csv(CSV_PATH, nrows=2000).to_csv(sample_path, index=False)
sample_path


WindowsPath('../data/interim/faf_sample_2000.csv')

## 11) PR#1 Summary (copy/paste into docs)
The dictionary below is the factual backbone for your PR#1 docs (`docs/…`). It captures:
- file size, rows, column count
- detected measure families + year coverage
- basic quality signals
- final star schema grain and measures


In [22]:
summary = {
    'csv_size_mb': float(round(csv_size_mb, 1)),
    'row_count_total': int(row_count),
    'n_columns': int(len(cols)),
    'measure_families_detected': {k: sorted(list(v)) for k, v in measure_map.items()},
    'duplicates_in_first_chunk': int(dup_count),
    'negatives_in_first_chunk_year_measures': int(neg_total),
    'key_cols_present': present_keys,
    'target_fact_grain': '(origin_zone_id, destination_zone_id, mode_id, commodity_id, trade_type_id, dist_band_id, year)',
    'fact_measures': ['tons','value','current_value','tmiles'],
}
summary


{'csv_size_mb': 663.0,
 'row_count_total': 2494901,
 'n_columns': 38,
 'measure_families_detected': {'tons': ['2018',
   '2019',
   '2020',
   '2021',
   '2022',
   '2023',
   '2024'],
  'value': ['2018', '2019', '2020', '2021', '2022', '2023', '2024'],
  'current_value': ['2018', '2019', '2020', '2021', '2022', '2023', '2024'],
  'tmiles': ['2018', '2019', '2020', '2021', '2022', '2023', '2024']},
 'duplicates_in_first_chunk': 0,
 'negatives_in_first_chunk_year_measures': 0,
 'key_cols_present': ['dms_orig',
  'dms_dest',
  'fr_orig',
  'fr_dest',
  'dms_mode',
  'fr_inmode',
  'fr_outmode',
  'sctg2',
  'trade_type',
  'dist_band'],
 'target_fact_grain': '(origin_zone_id, destination_zone_id, mode_id, commodity_id, trade_type_id, dist_band_id, year)',
 'fact_measures': ['tons', 'value', 'current_value', 'tmiles']}

## Next steps (project roadmap)

This PR intentionally stops at **understanding + modeling decisions**.

- **PR#2 — Build dimensions (Python):** create dimension tables from metadata (zones, modes, commodities, trade types, distance bands, year).
- **PR#3 — Staging load (raw CSV):** load raw rows to MySQL staging + log `stg_row_count` and ETL run metadata.
- **PR#4 — Wide → Long transform (memory-safe):** reshape `<measure_family>_<year>` into long form by year and load the fact table.
- **PR#4.5 — Validation gates:** expected long row counts, orphan checks, duplicate grain checks, negative/null checks.
- **PR#5+ — Indexing & analytics:** constraints, indexes tied to query patterns, EXPLAIN before/after, and business narrative queries.
